<a href="https://colab.research.google.com/github/sara-sgit/Plant-Disease-Classification/blob/main/Feature_Selection_3_Relief_F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ReliefF

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from sklearn.preprocessing import StandardScaler

Mounted at /content/drive


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from ReliefF import ReliefF
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import pickle as pk
import os


In [ ]:


with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/DenseNet/Leaky relu/ train_densenet_80_leakyrelu.pickle', 'rb') as f:
     x_tr0 = pk.load(f)

with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/DenseNet/Leaky relu/test_densenet_80_leakyrelu.pickle', 'rb') as f:
     x_ts0 = pk.load(f)


with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/Resnet/Leaky relu/ F_train_RESNET_Leakyrelu.pickle', 'rb') as f:
     x_tr1 = pk.load(f)

with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/Resnet/Leaky relu/F_test_RESNET_leakyrelu.pickle', 'rb') as f:
     x_ts1 = pk.load(f)


with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/vgg/LEAKY RELU/ right_features_training_LEAKYREL.pickle', 'rb') as f:
     x_tr2 = pk.load(f)

with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/vgg/LEAKY RELU/ right_features_test_LEAKYREL.pickle', 'rb') as f:
     x_ts2 = pk.load(f)



with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/labels/Y_train.pickle', 'rb') as f:
     y_tr = pk.load(f)

with open('/content/drive/MyDrive/BENNANI_AKRIB/FEATURES/labels/Y_test.pickle', 'rb') as f:
     y_ts = pk.load(f)


In [ ]:

#L'insruction de la fusion
x_tr= np.concatenate((x_tr0,x_tr1, x_tr2), axis=1)
x_ts= np.concatenate((x_ts0,x_ts1,x_ts2),axis=1)

In [ ]:

#making sure of data shape
print (x_tr.shape)
print(x_ts.shape
      )

(12811, 12288)
(3200, 12288)


In [ ]:

sc = StandardScaler()
X_train = sc.fit_transform(x_tr2)
X_test = sc.transform(x_ts2)

In [ ]:
fs = ReliefF(n_neighbors=20, n_features_to_keep=300)
X_train2 = fs.fit_transform(X_train, y_tr)
X_test2= fs.transform(X_test)


print("(No. of tuples, No. of Columns before ReliefF) : "+str(  X_train.shape)+
      "\n(No. of tuples, No. of Columns after ReliefF) : "+str(X_train2.shape))

#print(X_test2.shape)

In [ ]:
svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1, gamma='scale'))

# Fit the model to the trai ning data on the GPU
svm.fit(X_train2.astype(np.float32), y_tr.astype(np.float32))

# Predict the labels of the test data
y_pred = svm.predict(X_test2.astype(np.float32))

#________________________________   #########   #matrice de confusion    #############_____________________________________________________________

print('confusion matrix')
print(confusion_matrix(y_ts,y_pred))
labels=y_ts
sum((labels==y_pred)*1)/len(y_pred)

confusion matrix
[[414   1   2   0   5   0   0   3   0   0]
 [  9 144  18   0  14   6   6   2   1   0]
 [  1   7 360   3   5   1   3   1   0   1]
 [  0   2   1 171   8   3   2   2   1   0]
 [  3   2   9   3 332   1   2   0   2   0]
 [  0   0   0   1   1 314  14   1   2   0]
 [  1   2   4   1   1  15 253   0   1   3]
 [  1   3   0   0   1   2   1 634   0   0]
 [  0   0   0   2   2   0   0   0  71   0]
 [  0   0   0   0   1   0   5   0   0 312]]


0.9390625

#saving selected deep features

In [ ]:
pk.dump(X_train2,open('/content/drive/MyDrive/selecF/Relief/DenseNetLeaky/train_DenseNet_leakyr.pickle', 'wb'))
pk.dump(X_test2,open('/content/drive/MyDrive/selecF/Relief/DenseNetLeaky/test_DenseNet_leakyr.pickle', 'wb'))


#making sure that the same data is selected

In [ ]:
c=0
for i in range (4096):
  for j in  range(100):
    if X_train[1,i]==X_train2[1,j]:
      c=c+1
      print(f"this is X_train {X_train[1,i]} it's in position {i} which is = to the selected one {X_train2[1,j]} ")
print(c)